# My IRI-seq pipeline

This notebook is adapted from `IRIseq-pipeline-Chao-original.ipynb` for this server.

Key changes:
- Raw FASTQ input is read from `/p2/zulab/jtian/data/IRISeq/Data/`.
- All generated output goes under `/p2/zulab/jtian/data/IRISeq/output/`.
- Demultiplexing is skipped because the input data are already sample-level FASTQ files.
- Shell magics, `sbatch`, `squeue`, `ll`, `cat`, `head`, `zless`, `mkdir`, and `mv` cells are replaced by Python code.
- Local pipeline modules from `/p1/zulab_users/jtian/my_jupyter/IRI-seq/Module/` are copied into the output script folder with importable names.

Run with the `my_IRISeq_py38` kernel.

In [ ]:
from pathlib import Path
import csv
import gzip
import hashlib
import importlib
import re
import os
import shutil
import subprocess
import sys
from multiprocessing import Pool
from functools import partial

import numpy as np
import pandas as pd

PROJECT_DIR = Path('/p1/zulab_users/jtian/my_jupyter/IRI-seq')
RAW_DATA_DIR = Path('/p2/zulab/jtian/data/IRISeq/Data')
OUTPUT_DIR = Path('/p2/zulab/jtian/data/IRISeq/output')

SAMPLE_SHEET_DIR = OUTPUT_DIR / 'sample_sheets'
SCRIPT_DIR = OUTPUT_DIR / 'Scripts'
MODULE_DIR = SCRIPT_DIR / 'EasySpatial'
FASTQ_DIR = OUTPUT_DIR / 'prepared_fastq'
REPORT_DIR = OUTPUT_DIR / 'Report'

CDNA_OUTPUT = OUTPUT_DIR / 'cDNA'
BEAD_OUTPUT = OUTPUT_DIR / 'beads'
CONNECTION_OUTPUT = OUTPUT_DIR / 'connections'
UMAP_OUTPUT = CONNECTION_OUTPUT / 'UMAP'

CDNA_SAMPLES = ['old-cDNA', 'new-cDNA']
BEAD_SAMPLES = ['old-beads', 'new-beads']

# Barcode pickle files are bundled with this project.
PICKLE_DIR = PROJECT_DIR / 'IRIS' / 'pickle'
BARCODE_1_FILE = PICKLE_DIR / 'Spatial_R2_barcode_1.pickle'
BARCODE_2_FILE = PICKLE_DIR / 'Spatial_R2_barcode_2.pickle'
BARCODE_3_FILE = PICKLE_DIR / 'Spatial_R2_barcode_3.pickle'
BARCODE_4_FILE = PICKLE_DIR / 'Spatial_R2_barcode_4.pickle'
BARCODE_4_BEAD1_FILE = PICKLE_DIR / 'Spatial_R2_barcode_4_bead1.pickle'

# Fill these reference paths before running the cDNA alignment/gene-count section.
# The original notebook used a combined hg19/mm10 STAR index and a matching gene reference pickle.
REFERENCE_DIR = Path('/p2/zulab/jtian/data/IRISeq/reference')
STAR_INDEX = REFERENCE_DIR / 'STAR_hg19_mm10_RNAseq'
GENE_REFERENCE_PICKLE = REFERENCE_DIR / 'hg19_mm10_Gene_reference.pickle'

# Tools are searched from the active conda environment first.
STAR = shutil.which('STAR') or '/p1/zulab_users/jtian/anaconda3/envs/my_IRISeq_py38/bin/STAR'
SAMTOOLS = shutil.which('samtools') or '/p1/zulab_users/jtian/anaconda3/envs/my_IRISeq_py38/bin/samtools'
CUTADAPT = shutil.which('cutadapt') or '/p1/zulab_users/jtian/anaconda3/envs/my_IRISeq_py38/bin/cutadapt'

CORE_CDNA = 2
CORE_BEADS = 10
UMI_LIMIT = 50

for p in [OUTPUT_DIR, SAMPLE_SHEET_DIR, SCRIPT_DIR, MODULE_DIR, FASTQ_DIR, REPORT_DIR, CDNA_OUTPUT, BEAD_OUTPUT, CONNECTION_OUTPUT, UMAP_OUTPUT]:
    p.mkdir(parents=True, exist_ok=True)

print('RAW_DATA_DIR =', RAW_DATA_DIR)
print('OUTPUT_DIR   =', OUTPUT_DIR)
print('STAR         =', STAR)
print('SAMTOOLS     =', SAMTOOLS)
print('CUTADAPT     =', CUTADAPT)


## Python replacements for notebook shell commands

In [ ]:
def write_lines(path, lines):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text('\n'.join(lines) + '\n')
    return path


def file_size_gb(path):
    return Path(path).stat().st_size / 1024**3


def list_dir(path, pattern='*'):
    rows = []
    for p in sorted(Path(path).glob(pattern)):
        rows.append({
            'path': str(p),
            'type': 'dir' if p.is_dir() else 'file',
            'size_GB': None if p.is_dir() else round(file_size_gb(p), 3),
        })
    return pd.DataFrame(rows)


def fastq_head(path, n_records=2):
    path = Path(path)
    lines = []
    with gzip.open(path, 'rt') as f:
        for _ in range(n_records * 4):
            line = f.readline()
            if not line:
                break
            lines.append(line.rstrip('\n'))
    return lines


def count_fastq_reads(path):
    n_lines = 0
    with gzip.open(path, 'rt') as f:
        for _ in f:
            n_lines += 1
    return n_lines // 4


def count_gzip_lines(path):
    n_lines = 0
    with gzip.open(path, 'rt') as f:
        for _ in f:
            n_lines += 1
    return n_lines


def md5sum(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


def check_md5_file(md5_file):
    md5_file = Path(md5_file)
    rows = []
    with open(md5_file) as f:
        for line in f:
            if not line.strip():
                continue
            expected, rel = line.strip().split(maxsplit=1)
            rel = rel.lstrip('./')
            target = md5_file.parent / rel
            observed = md5sum(target)
            rows.append({'file': str(target), 'expected': expected, 'observed': observed, 'ok': expected == observed})
    return pd.DataFrame(rows)


def link_or_copy(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        return dst
    try:
        dst.symlink_to(src)
    except OSError:
        shutil.copy2(src, dst)
    return dst


def run_command(args, stdout_path=None):
    print('Running:', ' '.join(map(str, args)))
    if stdout_path is None:
        subprocess.run(list(map(str, args)), check=True)
    else:
        stdout_path = Path(stdout_path)
        stdout_path.parent.mkdir(parents=True, exist_ok=True)
        with open(stdout_path, 'w') as out:
            subprocess.run(list(map(str, args)), check=True, stdout=out)


## Check raw data and prepare pipeline input FASTQ names

The raw data are already demultiplexed FASTQ files, so this replaces the original `bcl2fastq` section. The pipeline expects names like `sample.R1.fastq.gz` and `sample.R2.fastq.gz`; the beads UMI step in the original code expects `sample.R3.fastq.gz`, so for this already-demultiplexed two-read dataset `R2` is linked as the beads `R3` input.

In [ ]:
raw_summary = list_dir(RAW_DATA_DIR)
display(raw_summary)

md5_table = check_md5_file(RAW_DATA_DIR / 'MD5.txt')
display(md5_table)
assert md5_table['ok'].all(), 'One or more raw FASTQ md5 checks failed.'

for sample in CDNA_SAMPLES:
    link_or_copy(RAW_DATA_DIR / sample / f'{sample}_R1.fq.gz', FASTQ_DIR / f'{sample}.R1.fastq.gz')
    link_or_copy(RAW_DATA_DIR / sample / f'{sample}_R2.fq.gz', FASTQ_DIR / f'{sample}.R2.fastq.gz')

for sample in BEAD_SAMPLES:
    link_or_copy(RAW_DATA_DIR / sample / f'{sample}_R1.fq.gz', FASTQ_DIR / f'{sample}.R1.fastq.gz')
    link_or_copy(RAW_DATA_DIR / sample / f'{sample}_R2.fq.gz', FASTQ_DIR / f'{sample}.R2.fastq.gz')
    link_or_copy(RAW_DATA_DIR / sample / f'{sample}_R2.fq.gz', FASTQ_DIR / f'{sample}.R3.fastq.gz')

CDNA_SAMPLE_FILE = write_lines(SAMPLE_SHEET_DIR / 'cDNA_samples.txt', CDNA_SAMPLES)
BEAD_SAMPLE_FILE = write_lines(SAMPLE_SHEET_DIR / 'bead_samples.txt', BEAD_SAMPLES)

list_dir(FASTQ_DIR)


In [ ]:
# Optional raw FASTQ preview, replacing shell `zless/head` checks.
for sample in CDNA_SAMPLES + BEAD_SAMPLES:
    print('\n#', sample, 'R1')
    print('\n'.join(fastq_head(FASTQ_DIR / f'{sample}.R1.fastq.gz', n_records=1)))
    print('\n#', sample, 'R2')
    print('\n'.join(fastq_head(FASTQ_DIR / f'{sample}.R2.fastq.gz', n_records=1)))


## Prepare local modules with importable names

The original notebook imported modules from another user's `EasySpatial` folder. This cell copies the local module files in this project into the output script directory with clean Python module names.

In [ ]:
module_map = {
    PROJECT_DIR / 'Module' / 'DNAcode' / 'Spatial_UMI_barcode_extraction (1).py': MODULE_DIR / 'Spatial_UMI_barcode_extraction.py',
    PROJECT_DIR / 'Module' / 'DNAcode' / 'Fastq_trim_multi_files (1).py': MODULE_DIR / 'Fastq_trim_multi_files.py',
    PROJECT_DIR / 'Module' / 'DNAcode' / 'Sam_rm_dup_barcode_UMI_multi_files (2).py': MODULE_DIR / 'Sam_rm_dup_barcode_UMI_multi_files.py',
    PROJECT_DIR / 'Module' / 'DNAcode' / 'Sam_gene_counting_multi_files (1).py': MODULE_DIR / 'Sam_gene_counting_multi_files.py',
    PROJECT_DIR / 'Module' / 'DNAcode' / 'Summary_gene_count_multi_files (1).py': MODULE_DIR / 'Summary_gene_count_multi_files.py',
    PROJECT_DIR / 'Module' / 'DNAcode' / 'Generate_adata (1).py': MODULE_DIR / 'Generate_adata.py',
    PROJECT_DIR / 'Module' / 'DNAcode' / 'File_functions (1).py': MODULE_DIR / 'File_functions.py',
    PROJECT_DIR / 'Module' / 'DNAcode' / 'Count_reads.py': MODULE_DIR / 'Count_reads.py',
    PROJECT_DIR / 'Module' / 'Beadcode' / 'UMI_barcode_extraction (2).py': MODULE_DIR / 'Bead_UMI_barcode_extraction.py',
    PROJECT_DIR / 'Module' / 'Beadcode' / 'spatial_barcode_extraction (3).py': MODULE_DIR / 'Bead_spatial_barcode_extraction.py',
    PROJECT_DIR / 'Module' / 'Beadcode' / 'Remove_duplicate_barcode (2).py': MODULE_DIR / 'Bead_Remove_duplicate_barcode.py',
}

for src, dst in module_map.items():
    if not src.exists():
        raise FileNotFoundError(src)
    text = src.read_text()
    if dst.name == 'Fastq_trim_multi_files.py':
        text = re.sub(r'cutadapt_path\s*=\s*"[^"]*cutadapt"', f'cutadapt_path = "{CUTADAPT}"', text)
    if dst.name == 'Bead_spatial_barcode_extraction.py':
        # Fix original script typo: it loaded a global list_barcode_4_file2 instead of the function argument.
        text = text.replace('with open(list_barcode_4_file2, \'rb\') as f:', 'with open(list_barcode_4_file, \'rb\') as f:')
    dst.write_text(text)

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import Spatial_UMI_barcode_extraction
import Sam_rm_dup_barcode_UMI_multi_files
import Sam_gene_counting_multi_files
import Summary_gene_count_multi_files
import Generate_adata
import Count_reads
import Bead_UMI_barcode_extraction
import Bead_spatial_barcode_extraction
import Bead_Remove_duplicate_barcode

print('Prepared modules in', MODULE_DIR)


## Pure-Python replacements for shell-heavy alignment/filter/trim steps

In [ ]:
def trim_fastq_files(input_folder, sample_file, output_folder, core, adapter_seq='AAAAAAAA', cutadapt_path=CUTADAPT):
    input_folder = Path(input_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    samples = [x.strip() for x in Path(sample_file).read_text().splitlines() if x.strip()]

    def trim_one(sample):
        run_command([
            cutadapt_path,
            '-a', adapter_seq,
            '--trim-n',
            '--minimum-length', '20',
            '-o', output_folder / f'{sample}.R2.fastq.gz',
            input_folder / f'{sample}.R2.fastq.gz',
        ])

    for sample in samples:
        trim_one(sample)


def run_star_alignment(input_folder, sample_file, output_folder, core_num=2, index=STAR_INDEX, star_path=STAR):
    input_folder = Path(input_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    index = Path(index)
    if not index.exists():
        raise FileNotFoundError(f'STAR index does not exist: {index}')
    samples = [x.strip() for x in Path(sample_file).read_text().splitlines() if x.strip()]

    for sample in samples:
        run_command([
            star_path,
            '--runThreadN', str(core_num),
            '--outSAMstrandField', 'intronMotif',
            '--genomeDir', index,
            '--readFilesCommand', 'zcat',
            '--readFilesIn', input_folder / f'{sample}.R2.fastq.gz',
            '--outFileNamePrefix', output_folder / sample,
            '--genomeLoad', 'LoadAndKeep',
        ])

    run_command([star_path, '--genomeDir', index, '--genomeLoad', 'Remove'])


def sam_filter_files(input_folder, sample_file, output_folder, core=2, samtools_path=SAMTOOLS):
    input_folder = Path(input_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    samples = [x.strip() for x in Path(sample_file).read_text().splitlines() if x.strip()]

    for sample in samples:
        input_sam = input_folder / f'{sample}Aligned.out.sam'
        output_sam = output_folder / f'{sample}.sam'
        print('Filtering', input_sam)
        p1 = subprocess.Popen([samtools_path, 'view', '-bh', '-q', '30', '-F', '4', str(input_sam)], stdout=subprocess.PIPE)
        p2 = subprocess.Popen([samtools_path, 'sort', '-@', str(core), '-'], stdin=p1.stdout, stdout=subprocess.PIPE)
        p1.stdout.close()
        with open(output_sam, 'w') as out:
            p3 = subprocess.Popen([samtools_path, 'view', '-h', '-'], stdin=p2.stdout, stdout=out)
            p2.stdout.close()
            rc3 = p3.wait()
        rc1 = p1.wait()
        rc2 = p2.wait()
        if rc1 or rc2 or rc3:
            raise RuntimeError(f'samtools filter failed for {sample}: {rc1}, {rc2}, {rc3}')


def summarize_read_counts(fastq_folder, sample_file, fastq_barcode_attached, sam_star, sam_rmdup, bed_gene_count, adata_folder, output_csv):
    samples = [x.strip() for x in Path(sample_file).read_text().splitlines() if x.strip()]
    df_count = pd.DataFrame({'Sample_name': samples})
    df_count['Count_Fastq_raw'] = Count_reads.Fastq_count_reads_files(str(fastq_folder), str(sample_file))
    df_tmp = Count_reads.Count_Align_STAR_files(str(sam_star), str(sample_file))
    df_count['Count_Fastq_filtered'] = Count_reads.Fastq_count_reads_files(str(fastq_barcode_attached), str(sample_file))
    df_count['Count_Fastq_trimmed'] = list(df_tmp[0])
    df_count['Count_Sam_mapped'] = list(df_tmp[1] + df_tmp[2])
    df_count['Count_Sam_unique'] = list(df_tmp[1])
    df_count['Count_Sam_rmdup'] = Count_reads.SAM_count_reads_files(str(sam_rmdup), str(sample_file))
    df_count['Count_Sam_annotated'] = Count_reads.count_mapped_reads_files(str(bed_gene_count), str(sample_file))
    df_count['Count_Cell_reads'] = Count_reads.Count_cell_reads(str(adata_folder))
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df_count.to_csv(output_csv, index=False)
    return df_count


## cDNA barcode attachment, trimming, alignment, gene counting, and AnnData generation

Before running this section, make sure `STAR_INDEX` and `GENE_REFERENCE_PICKLE` in the config cell point to real files on this server.

In [ ]:
def run_cdna_pipeline():
    if not Path(STAR_INDEX).exists():
        raise FileNotFoundError(f'Please set STAR_INDEX to a real STAR index directory: {STAR_INDEX}')
    if not Path(GENE_REFERENCE_PICKLE).exists():
        raise FileNotFoundError(f'Please set GENE_REFERENCE_PICKLE to a real gene annotation pickle: {GENE_REFERENCE_PICKLE}')

    fastq_barcode_attached = CDNA_OUTPUT / 'Fastq_barcode_attached'
    fastq_trimmed = CDNA_OUTPUT / 'Fastq_trimmed'
    sam_star = CDNA_OUTPUT / 'Sam_STAR'
    sam_filtered = CDNA_OUTPUT / 'Sam_filtered'
    sam_rmdup = CDNA_OUTPUT / 'Sam_rmdup'
    bed_gene_count = CDNA_OUTPUT / 'Bed_gene_count'
    summary_gene_count = CDNA_OUTPUT / 'Summary_gene_count'
    adata_folder = CDNA_OUTPUT / 'Adata'

    for p in [fastq_barcode_attached, fastq_trimmed, sam_star, sam_filtered, sam_rmdup, bed_gene_count, summary_gene_count, adata_folder]:
        p.mkdir(parents=True, exist_ok=True)

    Spatial_UMI_barcode_extraction.extract_spatial_barcode_files(
        str(FASTQ_DIR), str(CDNA_SAMPLE_FILE), str(fastq_barcode_attached), CORE_CDNA,
        str(BARCODE_1_FILE), str(BARCODE_2_FILE), str(BARCODE_3_FILE), str(BARCODE_4_BEAD1_FILE),
    )

    trim_fastq_files(fastq_barcode_attached, CDNA_SAMPLE_FILE, fastq_trimmed, CORE_CDNA, cutadapt_path=CUTADAPT)
    run_star_alignment(fastq_trimmed, CDNA_SAMPLE_FILE, sam_star, CORE_CDNA, STAR_INDEX, STAR)
    sam_filter_files(sam_star, CDNA_SAMPLE_FILE, sam_filtered, CORE_CDNA, SAMTOOLS)
    Sam_rm_dup_barcode_UMI_multi_files.rm_dup_files(str(sam_filtered), str(CDNA_SAMPLE_FILE), str(sam_rmdup), CORE_CDNA)
    Sam_gene_counting_multi_files.scRNA_count_parallel(str(sam_rmdup), str(CDNA_SAMPLE_FILE), str(bed_gene_count), str(GENE_REFERENCE_PICKLE), CORE_CDNA)
    Summary_gene_count_multi_files.Gene_count_summary(str(bed_gene_count), str(CDNA_SAMPLE_FILE), str(summary_gene_count))
    Generate_adata.generate_adata_from_gene_count(str(summary_gene_count), str(bed_gene_count / 'gene_anno.csv'), str(adata_folder), UMI_LIMIT)

    df_count = summarize_read_counts(
        FASTQ_DIR, CDNA_SAMPLE_FILE, fastq_barcode_attached, sam_star, sam_rmdup, bed_gene_count,
        adata_folder, adata_folder / 'df_reads.csv'
    )
    display(df_count)
    print('cDNA pipeline finished:', CDNA_OUTPUT)

# Uncomment to run cDNA pipeline after reference paths are configured.
# run_cdna_pipeline()


In [ ]:
# Python replacements for original output inspection cells.
adata_full_path = CDNA_OUTPUT / 'Adata' / 'adata_full.h5ad'
adata_exon_path = CDNA_OUTPUT / 'Adata' / 'adata_exon.h5ad'

if adata_full_path.exists():
    import scanpy as sc
    adata = sc.read_h5ad(adata_full_path)
    print(adata)
    display(adata.obs.head())
    display(adata.var.head())
else:
    print('cDNA AnnData output is not present yet:', adata_full_path)

for path in [
    CDNA_OUTPUT / 'Bed_gene_count' / f'{CDNA_SAMPLES[0]}.report',
    CDNA_OUTPUT / 'Bed_gene_count' / 'exon_anno.csv',
    CDNA_OUTPUT / 'Bed_gene_count' / 'gene_anno.csv',
]:
    if path.exists():
        print('\n#', path)
        display(pd.read_csv(path, nrows=10, header=None))


## Bead-bead interaction / spatial barcode pipeline

In [ ]:
def run_bead_pipeline():
    umi_attach = BEAD_OUTPUT / 'UMI_attach'
    spatial_barcode = BEAD_OUTPUT / 'Spatial_barcode_extraction'
    deduplicate_spatial = BEAD_OUTPUT / 'Spatial_barcode_rmdup'
    report_folder = BEAD_OUTPUT / 'report' / 'read_num_spatial_barcode'

    for p in [umi_attach, spatial_barcode, deduplicate_spatial, report_folder]:
        p.mkdir(parents=True, exist_ok=True)

    Bead_UMI_barcode_extraction.extract_spatial_barcode_files(
        str(FASTQ_DIR), str(BEAD_SAMPLE_FILE), str(umi_attach), CORE_BEADS,
        str(BARCODE_1_FILE), str(BARCODE_2_FILE), str(BARCODE_3_FILE), str(BARCODE_4_BEAD1_FILE),
    )
    Bead_spatial_barcode_extraction.extract_spatial_barcode_files(
        str(umi_attach), str(BEAD_SAMPLE_FILE), str(spatial_barcode), CORE_BEADS,
        str(BARCODE_1_FILE), str(BARCODE_2_FILE), str(BARCODE_3_FILE), str(BARCODE_4_FILE),
    )
    Bead_Remove_duplicate_barcode.remove_duplicates_files(
        str(spatial_barcode), str(BEAD_SAMPLE_FILE), str(deduplicate_spatial), CORE_BEADS,
    )

    rows = []
    for sample in BEAD_SAMPLES:
        rows.append({
            'sample': sample,
            'total_reads': count_fastq_reads(FASTQ_DIR / f'{sample}.R2.fastq.gz'),
            'Filtering_bead1_barcode': count_fastq_reads(umi_attach / f'{sample}.R2.fastq.gz'),
            'Filtering_bead2_barcode': count_gzip_lines(spatial_barcode / f'{sample}.spatial.txt.gz'),
            'Remove_duplicates': count_gzip_lines(deduplicate_spatial / f'{sample}.spatial.csv.gz'),
        })
    read_number = pd.DataFrame(rows)
    read_number.to_csv(report_folder / 'read_number.csv', index=False)
    display(read_number)
    print('Bead pipeline finished:', BEAD_OUTPUT)

# Uncomment to run bead interaction pipeline.
# run_bead_pipeline()


In [ ]:
read_number_csv = BEAD_OUTPUT / 'report' / 'read_num_spatial_barcode' / 'read_number.csv'
if read_number_csv.exists():
    import matplotlib.pyplot as plt
    df_read_numb = pd.read_csv(read_number_csv)
    df_numeric = df_read_numb.drop(columns=['sample']).apply(pd.to_numeric, errors='coerce')
    df_ratio = df_numeric.div(df_numeric.iloc[:, 0], axis=0)
    column_avg = df_ratio.mean()
    display(df_read_numb)

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(column_avg.index, column_avg.values)
    ax.plot(column_avg.index, column_avg.values, color='orange')
    ax.set_ylabel('Ratio of total reads')
    ax.tick_params(axis='x', rotation=90)
    plt.show()
else:
    print('Bead read-number report is not present yet:', read_number_csv)


## Scanpy analysis for cDNA AnnData

In [ ]:
def run_scanpy_cdna_analysis(sample_to_keep=None):
    import scanpy as sc
    import matplotlib.pyplot as plt

    adata = sc.read_h5ad(CDNA_OUTPUT / 'Adata' / 'adata_full.h5ad')

    def extract_sample(cell_name):
        for sample in CDNA_SAMPLES:
            if sample in cell_name:
                return sample
        return None

    adata.obs['sample_source'] = adata.obs.index.to_series().apply(extract_sample)
    adata.obs['extracted_ID'] = adata.obs.index.to_series().str.split('.').str[-1].apply(lambda x: x.replace('-', ''))
    adata.var['Mouse_gene'] = adata.var.index.str[:4] == 'ENSM'
    adata = adata[:, adata.var['Mouse_gene'] == True].copy()
    adata.obs['n_genes'] = (adata.X > 0).sum(axis=1)
    adata.var.index = adata.var['Gene_name'].astype(str)
    adata.var_names_make_unique()

    if sample_to_keep is not None:
        adata = adata[adata.obs['sample_source'] == sample_to_keep, :].copy()

    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=5, min_disp=0.5)
    adata.raw = adata
    adata = adata[:, adata.var.highly_variable].copy()
    regressors = [x for x in ['total_counts', 'pct_counts_mt'] if x in adata.obs.columns]
    if regressors:
        sc.pp.regress_out(adata, regressors)
    sc.tl.pca(adata)
    sc.pp.scale(adata, max_value=10)
    sc.pp.neighbors(adata, n_neighbors=20, n_pcs=25)
    sc.tl.umap(adata, min_dist=0, spread=3)
    sc.tl.leiden(adata, resolution=1.3)
    sc.pl.umap(adata, color=['leiden'])
    return adata

# Uncomment after cDNA pipeline has generated adata_full.h5ad.
# adata = run_scanpy_cdna_analysis()


## Spatial interaction matrix and UMAP from bead outputs

In [ ]:
def bead1_bead2_interaction(input_df):
    inside_df = input_df.copy()
    inside_df.columns = ['Bead1_seq', 'UMI', 'Bead2_seq']
    n_umi_per_interaction = inside_df.groupby(['Bead1_seq', 'Bead2_seq']).size().reset_index(name='n_umi')
    n_umi_per_bead1 = n_umi_per_interaction.groupby('Bead1_seq').size().reset_index(name='n_umi')
    return {'n_umi_per_interaction': n_umi_per_interaction, 'n_umi_per_bead1': n_umi_per_bead1}


def build_connection_matrix(sample, log10_umi_threshold=0.8):
    input_path = BEAD_OUTPUT / 'Spatial_barcode_rmdup' / f'{sample}.spatial.csv.gz'
    df = pd.read_csv(input_path, compression='gzip')
    df.columns = ['Bead1_seq', 'UMI', 'Bead2_seq']
    result = bead1_bead2_interaction(df)
    interactions = result['n_umi_per_interaction']
    interactions['log_transformed'] = np.log10(interactions['n_umi'])
    filtered = interactions[interactions['log_transformed'] >= log10_umi_threshold]
    matrix = filtered.pivot_table(index='Bead1_seq', columns='Bead2_seq', values='n_umi', fill_value=0, aggfunc='sum')
    output_csv = UMAP_OUTPUT / f'{sample}_connection_matrix.csv'
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    matrix.to_csv(output_csv)
    print('Saved', output_csv, 'shape=', matrix.shape)
    return matrix


def run_connection_umap(sample, n_components=600):
    import matplotlib.pyplot as plt
    import umap
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler

    matrix = build_connection_matrix(sample)
    matrix_log1p = np.log1p(matrix)
    matrix_standardized = StandardScaler().fit_transform(matrix_log1p)
    n_components = min(n_components, matrix_standardized.shape[0], matrix_standardized.shape[1])
    matrix_pca = PCA(n_components=n_components).fit_transform(matrix_standardized)
    coords = umap.UMAP(n_neighbors=25, min_dist=0.2, metric='euclidean', random_state=42, n_epochs=500, verbose=True, spread=0.6).fit_transform(matrix_pca)
    umap_df = pd.DataFrame(coords, columns=['UMAP1', 'UMAP2'], index=matrix.index)
    output_csv = UMAP_OUTPUT / f'{sample}_connection_umap.csv'
    umap_df.to_csv(output_csv)
    plt.figure(figsize=(8, 8))
    plt.scatter(coords[:, 0], coords[:, 1], s=1)
    plt.gca().set_aspect('equal', 'datalim')
    plt.show()
    print('Saved', output_csv)
    return umap_df

# Uncomment after bead pipeline has generated Spatial_barcode_rmdup outputs.
# connection_umap = run_connection_umap(BEAD_SAMPLES[0])


## Merge cDNA clusters with bead-derived spatial UMAP

In [ ]:
def plot_cdna_clusters_on_connection_umap(adata, connection_umap_csv):
    import matplotlib.pyplot as plt
    import scanpy as sc

    df = pd.read_csv(connection_umap_csv)
    adata.obs['cell_name_temp'] = adata.obs.index
    merged = adata.obs.merge(df, left_on='extracted_ID', right_on='Unnamed: 0', how='left', suffixes=('_adata', '_df'))
    merged.index = merged['cell_name_temp']
    adata.obs = merged.drop(columns=['cell_name_temp', 'Unnamed: 0'], errors='ignore')

    colors = sc.pl.palettes.default_102
    cluster_colors = {cluster: colors[i] for i, cluster in enumerate(adata.obs['leiden'].cat.categories)}
    colors_mapped = adata.obs['leiden'].map(cluster_colors).tolist()

    plt.figure(figsize=(10, 8))
    plt.scatter(adata.obs['UMAP1'], adata.obs['UMAP2'], c=colors_mapped, s=20, alpha=0.7)
    labels = list(adata.obs['leiden'].cat.categories)
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=cluster_colors[cat], markersize=10) for cat in labels]
    plt.legend(handles=handles, labels=labels, loc='upper right', fontsize='small', title='Leiden Clusters')
    plt.title('Connection UMAP colored by cDNA Leiden clusters')
    plt.xlabel('UMAP1')
    plt.ylabel('UMAP2')
    plt.grid(True)
    plt.show()

# Example after both analyses are available:
# plot_cdna_clusters_on_connection_umap(adata, UMAP_OUTPUT / f'{BEAD_SAMPLES[0]}_connection_umap.csv')
